# 09 Re-Analysis with ATAC Data

In [2]:
# Initialize
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun

docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

plotHeatmap 3.5.6
bedtools v2.31.1


In [3]:
# First make a associative array to store bigWig and peak locations for each sample
fpath="source_data/atac_bw/"

# D8 ATAC
declare -A bigWigFiles
for group in GFPpos GFPneg CX3CR1pos; do
    for target in shCD19 shRunx3; do
        fn=${group}_${target}.bw
        bigWigFiles[${group}_${target}]="${fpath}${fn}"
    done
done

#D5 ATAC by ST
for st in D5_shCD19 D5_shRunx3; do
    fn=${st}.bw
    bigWigFiles[${st}]="${fpath}${fn}"
done


# Compute Matrix
mkdir -p 09_atac

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["D5_shCD19"]} \
    ${bigWigFiles["D5_shRunx3"]} \
    ${bigWigFiles["GFPpos_shCD19"]} \
    ${bigWigFiles["GFPpos_shRunx3"]} \
    ${bigWigFiles["GFPneg_shCD19"]} \
    ${bigWigFiles["GFPneg_shRunx3"]} \
    ${bigWigFiles["CX3CR1pos_shCD19"]} \
    ${bigWigFiles["CX3CR1pos_shRunx3"]} \
    -R \
    "01_peakEDA/cluster1.fullpeak.clean.bed" \
    "01_peakEDA/cluster2.fullpeak.clean.bed" \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 48 \
    --sortUsing mean \
    -out "09_atac/atac.matrix.gz"

deeptools plotHeatmap \
    -m "09_atac/atac.matrix.gz" \
    -out "09_atac/atac.matrix.pdf" \
    --sortUsing mean \
    --colorMap  RdYlBu_r RdYlBu_r\
                RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r \
    --samplesLabel  "D5_shCD19" "D5_shRunx3" \
                    "GFPpos_shCd19" "GFPpos_shRunx3" \
                    "GFPneg_shCd19" "GFPneg_shRunx3" \
                    "CX3CR1pos_shCd19" "CX3CR1pos_shRunx3" \
    --regionsLabel "Type1" "Type2"